# Decision Learning Tree 

**Author: Rownak Deb Kabya -22400196**


## Introduction

This Jupyter notebook implements a Decision Learning Tree without using any external libraries for binary attributes. The decision tree is built using a recursive algorithm that selects the best attribute to split the data at each node based on information gain. Then we will visualize the decision tree using a Graphviz.

#### Decision tree learning
The pseudo-code for the decision tree learning algorithm is as follows:
```python
function DT-LEARNING(examples, attributes, parent examples)
    if empty(examples) then return PLURALITY-VAL(parent examples)
    else if all examples have same classification then return the classification
    else if empty(attributes) then return PLURALITY-VAL(examples)
    else
        A ← argmax
            a∈attributes
        IMPORTANCE(a, examples)
        tree ← a new decision tree with root test A
        for vk ∈ A do
            exs ← {e|e ∈ examples ∧ e.A = vk}
            subtree ← DT-LEARNING(exs, attributes − A, examples)
            add a branch to tree with label (A = vk) and subtree subtree
        end for
        return tree
    end if
end function
```

### Training Set
We will first create a training set to test our decision tree learning algorithm. The training set will consist of binary attributes.

In [1]:
import pandas as pd 

data = [
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 0, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 0, 'Like': 1, 'temp': 0, 'fun': 0},
    {'cloud': 0, 'rain': 0, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 0, 'temp': 1, 'fun': 0},
    {'cloud': 1, 'rain': 0, 'Like': 0, 'temp': 1, 'fun': 0},
    {'cloud': 0, 'rain': 1, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 0, 'fun': 1},
    {'cloud': 0, 'rain': 0, 'Like': 1, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 0, 'rain': 1, 'Like': 1, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 1, 'fun': 0},
    {'cloud': 1, 'rain': 0, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 0, 'rain': 1, 'Like': 0, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 0, 'Like': 0, 'temp': 1, 'fun': 0},
    {'cloud': 0, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 0, 'rain': 0, 'Like': 1, 'temp': 0, 'fun': 1},
    {'cloud': 1, 'rain': 0, 'Like': 1, 'temp': 0, 'fun': 0},
    {'cloud': 0, 'rain': 1, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 0, 'temp': 1, 'fun': 0},
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 0, 'rain': 1, 'Like': 1, 'temp': 0, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 0, 'fun': 1},
    {'cloud': 0, 'rain': 0, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 0, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 0, 'rain': 1, 'Like': 0, 'temp': 1, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1}
]

df = pd.DataFrame(data, columns=['cloud', 'rain', 'Like', 'temp', 'fun'])

df

,cloud,rain,Like,temp,fun
0,0,0,0,0,0
1,0,1,1,1,1
2,1,1,1,1,1
3,1,0,1,0,0
4,0,0,1,1,1
5,1,1,0,1,0
6,1,0,0,1,0
7,0,1,0,0,0
8,1,1,1,0,1
9,0,0,1,0,0


### Plurality Value
This function returns the most common class label.

In [2]:
def plurality_value(examples, target_attribute):
    return examples[target_attribute].mode()[0]

### Entropy Calculation
Entropy is a measure of uncertainty or impurity in a dataset. It is calculated using the formula:
$ H(S) = -\sum_{i=1}^{c} p_i \log_2(p_i) $

where $ p_i $ is the proportion of examples in class $ i $ and $ c $ is the number of classes.

In [ ]:
import numpy as np
def entropy(examples, target_attribute):
    values, counts = np.unique(examples[target_attribute], return_counts=True) # Get unique values and their counts
    probabilities = counts / counts.sum() # Calculate probabilities
    return -np.sum(probabilities * np.log2(probabilities + 1e-9)) # Calculate entropy and return it 

### Information Gain

Information gain tells us which attribute splits the data best.
It is calculated using the formula:
$ IG(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} H(S_v) $

In [4]:
def information_gain(examples, attribute, target_attribute):
    total_entropy = entropy(examples, target_attribute)
    values = examples[attribute].unique()
    weighted_entropy = 0
    for v in values:
        subset = examples[examples[attribute] == v]
        weighted_entropy += (len(subset) / len(examples)) * entropy(subset, target_attribute)
    return total_entropy - weighted_entropy

### Decision Tree Node Class

This class represents each node (attribute split or leaf) in the tree.

In [5]:
class TreeNode:
    def __init__(self, attribute=None, is_leaf=False, classification=None):
        self.attribute = attribute
        self.is_leaf = is_leaf
        self.classification = classification
        self.children = {}  # value: TreeNode

    def add_child(self, value, node):
        self.children[value] = node

### Implementation of the Decision Tree Learning Algorithm
This is the core of the algorithm, directly following the pseudocode.

In [6]:
def dt_learning(examples, attributes, parent_examples, target_attribute):
    if len(examples) == 0:
        return TreeNode(is_leaf=True, classification=plurality_value(parent_examples, target_attribute))
    elif len(examples[target_attribute].unique()) == 1:
        return TreeNode(is_leaf=True, classification=examples[target_attribute].iloc[0])
    elif len(attributes) == 0:
        return TreeNode(is_leaf=True, classification=plurality_value(examples, target_attribute))
    else:
        # Compute information gain for each attribute
        gains = {a: information_gain(examples, a, target_attribute) for a in attributes}
        A = max(gains, key=gains.get)
        node = TreeNode(attribute=A)
        for v in sorted(examples[A].unique()):
            subset = examples[examples[A] == v]
            remaining_attributes = [attr for attr in attributes if attr != A]
            child = dt_learning(subset, remaining_attributes, examples, target_attribute)
            node.add_child(v, child)
        return node

### Training the Decision Tree
We will train the decision tree using the training set we created earlier.

In [7]:
attributes = ['cloud', 'rain', 'Like', 'temp']
target_attribute = 'fun'
tree = dt_learning(df, attributes, df, target_attribute)

### Visualization of the Decision Tree

we will visualize the tree using Graphviz.

In [8]:
from graphviz import Digraph

def render_tree(node, dot=None, parent=None, edge_label=''):
    if dot is None:
        dot = Digraph()
    node_id = str(id(node))
    if node.is_leaf:
        dot.node(node_id, f"Leaf: {node.classification}")
    else:
        dot.node(node_id, f"{node.attribute}")
    if parent is not None:
        dot.edge(parent, node_id, label=str(edge_label))
    for attr_value, child in node.children.items():
        render_tree(child, dot, node_id, edge_label=attr_value)
    return dot


### Predict with the Tree

We will now predict with the trained decision tree.

In [9]:
def predict(tree, instance):
    while not tree.is_leaf:
        value = instance[tree.attribute]
        if value in tree.children:
            tree = tree.children[value]
        else:
            return None  # Unknown value
    return tree.classification

sample = {'cloud': 0, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1}
print(f"Prediction for {sample}: {predict(tree, sample)}")

Prediction for {'cloud': 0, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1}: 1


### Graphviz Visualization

In [10]:
from IPython.display import Image, display

# Render and display in the notebook
dot.render('decision_tree', format='png')
display(Image(filename='decision_tree.png'))

NameError: name 'dot' is not defined

## Conclusion
In this notebook, we implemented a Decision Learning Tree from scratch for binary attributes. We built the decision tree using a recursive algorithm based on information gain and visualized it using Graphviz. The decision tree can be used to make predictions on new data points based on the learned structure.